# Real Data Pipeline - Social Media Analytics

This notebook demonstrates how to analyze social media data from Twitter, Reddit, and news sources using the real data pipeline.

## Setup

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("SocialMediaAnalytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0,org.postgresql:postgresql:42.7.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

## 1. Load Data from Delta Lake

In [ ]:
# Load social media data from silver layer
try:
    twitter_df = spark.read.format("delta").load("/home/jovyan/data/silver/twitter")
    print(f"Twitter data loaded: {twitter_df.count()} records")
    twitter_df.printSchema()
except Exception as e:
    print(f"Could not load Twitter data: {e}")
    twitter_df = None

In [ ]:
try:
    reddit_df = spark.read.format("delta").load("/home/jovyan/data/silver/reddit")
    print(f"Reddit data loaded: {reddit_df.count()} records")
    reddit_df.printSchema()
except Exception as e:
    print(f"Could not load Reddit data: {e}")
    reddit_df = None

In [ ]:
try:
    news_df = spark.read.format("delta").load("/home/jovyan/data/silver/news")
    print(f"News data loaded: {news_df.count()} records")
    news_df.printSchema()
except Exception as e:
    print(f"Could not load News data: {e}")
    news_df = None

## 2. Twitter Analytics

In [ ]:
if twitter_df:
    # Basic Twitter statistics
    twitter_stats = twitter_df.agg(
        count("*").alias("total_tweets"),
        countDistinct("user_id").alias("unique_users"),
        avg("sentiment_score").alias("avg_sentiment"),
        avg("engagement_rate").alias("avg_engagement"),
        countDistinct("language").alias("languages")
    ).collect()[0]
    
    print("📊 Twitter Statistics:")
    print(f"Total Tweets: {twitter_stats['total_tweets']:,}")
    print(f"Unique Users: {twitter_stats['unique_users']:,}")
    print(f"Average Sentiment: {twitter_stats['avg_sentiment']:.4f}")
    print(f"Average Engagement: {twitter_stats['avg_engagement']:.4f}")
    print(f"Languages: {twitter_stats['languages']}")

In [ ]:
if twitter_df:
    # Top hashtags analysis
    hashtags_df = twitter_df.select(
        explode(col("hashtags")).alias("hashtag")
    ).groupBy("hashtag") \
     .agg(count("*").alias("frequency")) \
     .orderBy(desc("frequency")) \
     .limit(20)
    
    hashtags_pd = hashtags_df.toPandas()
    
    # Visualize top hashtags
    fig = px.bar(
        hashtags_pd, 
        x='frequency', 
        y='hashtag', 
        orientation='h',
        title='Top 20 Hashtags',
        labels={'frequency': 'Frequency', 'hashtag': 'Hashtag'}
    )
    fig.update_layout(height=600)
    fig.show()

In [ ]:
if twitter_df:
    # Sentiment analysis over time
    sentiment_trend = twitter_df.select(
        to_date(col("created_at")).alias("date"),
        col("sentiment_score")
    ).groupBy("date") \
     .agg(
         avg("sentiment_score").alias("avg_sentiment"),
         count("*").alias("tweet_count")
     ).orderBy("date")
    
    sentiment_pd = sentiment_trend.toPandas()
    
    # Create subplot with sentiment and volume
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Daily Average Sentiment', 'Daily Tweet Volume'),
        vertical_spacing=0.1
    )
    
    fig.add_trace(
        go.Scatter(
            x=sentiment_pd['date'], 
            y=sentiment_pd['avg_sentiment'],
            mode='lines+markers',
            name='Sentiment',
            line=dict(color='blue')
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(
            x=sentiment_pd['date'], 
            y=sentiment_pd['tweet_count'],
            name='Tweet Count',
            marker_color='lightblue'
        ),
        row=2, col=1
    )
    
    fig.update_layout(height=600, title_text="Twitter Sentiment and Volume Analysis")
    fig.show()

## 3. Reddit Analytics

In [ ]:
if reddit_df:
    # Subreddit activity analysis
    subreddit_activity = reddit_df.groupBy("subreddit") \
        .agg(
            count("*").alias("post_count"),
            avg("score").alias("avg_score"),
            avg("sentiment_score").alias("avg_sentiment"),
            sum("num_comments").alias("total_comments")
        ).orderBy(desc("post_count")) \
         .limit(15)
    
    subreddit_pd = subreddit_activity.toPandas()
    
    # Visualize subreddit activity
    fig = px.scatter(
        subreddit_pd,
        x='avg_score',
        y='post_count',
        size='total_comments',
        color='avg_sentiment',
        hover_name='subreddit',
        title='Subreddit Activity: Score vs Post Count',
        labels={
            'avg_score': 'Average Score',
            'post_count': 'Post Count',
            'avg_sentiment': 'Average Sentiment'
        }
    )
    fig.show()

## 4. News Analytics

In [ ]:
if news_df:
    # News sentiment by category
    news_category = news_df.groupBy("category") \
        .agg(
            count("*").alias("article_count"),
            avg("sentiment_score").alias("avg_sentiment"),
            countDistinct("source").alias("source_count")
        ).orderBy(desc("article_count"))
    
    news_pd = news_category.toPandas()
    
    # Visualize news sentiment by category
    fig = px.bar(
        news_pd,
        x='category',
        y='article_count',
        color='avg_sentiment',
        title='News Articles by Category and Sentiment',
        labels={
            'category': 'Category',
            'article_count': 'Article Count',
            'avg_sentiment': 'Average Sentiment'
        }
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

## 5. Cross-Platform Analysis

In [ ]:
# Compare sentiment across platforms
platform_sentiment = []

if twitter_df:
    twitter_sent = twitter_df.agg(avg("sentiment_score").alias("sentiment")).collect()[0]['sentiment']
    platform_sentiment.append({'platform': 'Twitter', 'sentiment': twitter_sent})

if reddit_df:
    reddit_sent = reddit_df.agg(avg("sentiment_score").alias("sentiment")).collect()[0]['sentiment']
    platform_sentiment.append({'platform': 'Reddit', 'sentiment': reddit_sent})

if news_df:
    news_sent = news_df.agg(avg("sentiment_score").alias("sentiment")).collect()[0]['sentiment']
    platform_sentiment.append({'platform': 'News', 'sentiment': news_sent})

if platform_sentiment:
    sentiment_comparison = pd.DataFrame(platform_sentiment)
    
    fig = px.bar(
        sentiment_comparison,
        x='platform',
        y='sentiment',
        title='Average Sentiment Across Platforms',
        color='sentiment',
        color_continuous_scale='RdYlGn'
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Neutral")
    fig.show()

## 6. Real-time Data Quality Checks

In [ ]:
# Data quality summary
print("📊 Data Quality Summary:")
print("=" * 50)

if twitter_df:
    twitter_quality = twitter_df.select(
        count("*").alias("total"),
        sum(when(col("sentiment_score").isNull(), 1).otherwise(0)).alias("null_sentiment"),
        sum(when(length(col("text")) < 10, 1).otherwise(0)).alias("short_text"),
        countDistinct("id").alias("unique_ids")
    ).collect()[0]
    
    print(f"Twitter:")
    print(f"  Total records: {twitter_quality['total']:,}")
    print(f"  Unique IDs: {twitter_quality['unique_ids']:,}")
    print(f"  Null sentiment: {twitter_quality['null_sentiment']:,}")
    print(f"  Short text (<10 chars): {twitter_quality['short_text']:,}")
    print(f"  Duplicate rate: {((twitter_quality['total'] - twitter_quality['unique_ids']) / twitter_quality['total'] * 100):.2f}%")

if reddit_df:
    reddit_quality = reddit_df.select(
        count("*").alias("total"),
        sum(when(col("sentiment_score").isNull(), 1).otherwise(0)).alias("null_sentiment"),
        sum(when(col("score") < 0, 1).otherwise(0)).alias("negative_score"),
        countDistinct("id").alias("unique_ids")
    ).collect()[0]
    
    print(f"\nReddit:")
    print(f"  Total records: {reddit_quality['total']:,}")
    print(f"  Unique IDs: {reddit_quality['unique_ids']:,}")
    print(f"  Null sentiment: {reddit_quality['null_sentiment']:,}")
    print(f"  Negative score posts: {reddit_quality['negative_score']:,}")
    print(f"  Duplicate rate: {((reddit_quality['total'] - reddit_quality['unique_ids']) / reddit_quality['total'] * 100):.2f}%")

## 7. Export Results

In [ ]:
# Save analysis results
output_path = "/home/jovyan/data/analysis_results"

if twitter_df:
    # Save daily Twitter aggregates
    twitter_daily = twitter_df.select(
        to_date(col("created_at")).alias("date"),
        col("sentiment_score"),
        col("engagement_rate")
    ).groupBy("date") \
     .agg(
         count("*").alias("tweet_count"),
         avg("sentiment_score").alias("avg_sentiment"),
         avg("engagement_rate").alias("avg_engagement")
     )
    
    twitter_daily.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"{output_path}/twitter_daily_analysis.csv")
    
    print("✅ Twitter analysis saved to CSV")

print("\n📁 Analysis complete! Check the output directory for results.")

In [ ]:
# Clean up
spark.stop()